In [12]:
from datasets import load_dataset

In [20]:
dataset=load_dataset("csv",data_files={"train":"s3://llmfine/dataset/pharma_instruction_data.csv"},split="train")

In [21]:
dataset


Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [10]:
!pip install cmake==3.27.0 pyarrow==16.1.0 datasets transformers accelerate sagemaker boto3 --upgrade --no-cache-dir

INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 376.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 493.8 MB/s  0:00:00
  Attempting uninstall: botocore
    Found existing installation: botocore 1.43.86
    Uninstalling botocore-1.43.86:
      Successfully uninstalled botocore-1.43.86
  Attempting uninstall: boto3━━━━━━━━━━━━━━━━━━━ 0/7 [botocore]
    Found existing installation: boto3 1.43.860m 0/7 [botocore]
    Uninstalling

In [22]:
print(dataset[0])

{'instruction': 'Explain the mechanism of action of Metformin.', 'input': None, 'output': 'Metformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'}


In [25]:
def format_example(example):
    prompt=f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text":prompt}

dataset=dataset.map(format_example)
dataset





Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 5
})

In [32]:
dataset["text"][0]
import sys
!{sys.executable} -m pip install peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 42.3 MB/s  0:00:00


In [48]:
from transformers import AutoTokenizer,AutoModelForCausalLM,TrainingArguments,Trainer
from peft import LoraConfig,get_peft_model,TaskType

model_id="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [49]:
tokenizer=AutoTokenizer.from_pretrained(model_id)
tokenizer.eos_token


'</s>'

In [50]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    

In [52]:
def tokenize_fn(example):
    tokens=tokenizer(example["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"]=tokens["input_ids"].copy()
    return tokens

tokenized=dataset.map(tokenize_fn,batched=True)    
    

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [53]:
from peft import LoraConfig, get_peft_model, TaskType

In [54]:
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

/home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/torch/cuda/__init__.py:1074: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


RuntimeError: unable to mmap 4400216536 bytes from file </home/ec2-user/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-intermediate-step-1431k-3T/snapshots/59f6f375b26bde864a6ca194a9a3044570490064/model.safetensors>: Cannot allocate memory (12)

In [55]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16, #scaling factor
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [57]:
model = get_peft_model(model, lora_config)

NameError: name 'model' is not defined

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

In [ ]:
trainer.train()

In [ ]:

model_path = "/content/tinyllama-instruction/checkpoint-3"

In [ ]:

instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:

prompt = "Explain the mechanism of action of Metformin."

In [ ]:

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:

outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [58]:
import psutil

ram = psutil.virtual_memory()

print("Total RAM:", ram.total / 1024**3, "GB")
print("Available RAM:", ram.available / 1024**3, "GB")

Total RAM: 3.744640350341797 GB
Available RAM: 0.5488815307617188 GB
